# 07 — Export: `player_advanced_stats.json` (Phase 7)

Produces `data/output/player_advanced_stats.json` for the dashboard: this
league's custom model's projection (point/floor/ceiling), a trailing usage
snapshot, and a season-long xFP/luck summary, keyed by Sleeper player_id.

All the real logic lives in `src/export.py` — this notebook orchestrates it
and reports what actually happened, per `CLAUDE.md`'s "reusable logic goes in
`src/`, notebooks stay single-purpose."

Predicting an unplayed week reuses the existing point-in-time-safe feature
pipeline unmodified (see `src/export.py`'s module docstring for the full
mechanism) — no new feature logic, just a correctly-shaped stub row for the
target week fed through `add_context_features`/`add_rolling_features`.

In [ ]:
import sys
from pathlib import Path
import json
import subprocess

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.ingest import (
    DATA_OUTPUT, DEFAULT_LEAGUE_ID,
    get_id_crosswalk, get_pbp, get_schedule, get_sleeper_league, get_sleeper_players, get_sleeper_rosters,
)
from src.export import (
    build_target_week_features, build_usage_snapshot, build_trend_snapshot, build_xfp_summary,
    build_weekly_xfp, build_radar_snapshot, build_heatmap_snapshot, get_export_candidates, get_export_scope,
    predict_target_week, assemble_player_advanced_stats, validate_export,
)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 12)

## Configuration

Target week is the upcoming, not-yet-played week — Week 1 of the 2026
season. `SEASONS_TRAINED` matches every other model in this pipeline
(2018-2025, the default established after the Phase 6 data-volume result).
`LEAGUE_ID` is `DEFAULT_LEAGUE_ID`, which points at the 2026 league.

In [2]:
TARGET_SEASON = 2026
TARGET_WEEK = 1
SEASONS_TRAINED = list(range(2018, 2026))
XFP_SEASON = TARGET_SEASON - 1  # most recently COMPLETED season -- 2026 has no games played yet
LEAGUE_ID = DEFAULT_LEAGUE_ID
TOP_N_FREE_AGENTS = 300

MODEL_VERSION = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], cwd=PROJECT_ROOT, capture_output=True, text=True
).stdout.strip() or "unknown"
print("MODEL_VERSION:", MODEL_VERSION)
print("target:", TARGET_SEASON, "week", TARGET_WEEK)

MODEL_VERSION: 491fa5c
target: 2026 week 1


## Load historical data

Reads the already-built `weekly_features.parquet` (from `03_usage_features.ipynb`)
rather than rebuilding it — this notebook is about export, not feature
engineering. Also pulls the real 2026 schedule, Sleeper's live player DB, the
ID crosswalk, and this league's real league object.

In [3]:
weekly_features = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "weekly_features.parquet")
print(f"weekly_features: {len(weekly_features):,} rows, seasons {weekly_features['season'].min()}-{weekly_features['season'].max()}")

# Schedule, league status, and rosters are LIVE state, not historical fact --
# refreshed every run rather than trusted from a cache that could be stale by
# the time this notebook is re-run (a draft could have happened since, lines
# could have moved).
schedule_2026 = get_schedule([TARGET_SEASON], refresh=True)
wk1_games = schedule_2026[(schedule_2026["week"] == TARGET_WEEK) & (schedule_2026["game_type"] == "REG")]
print(f"\n{TARGET_SEASON} schedule: {len(schedule_2026)} rows, {len(wk1_games)} week-{TARGET_WEEK} games")
print(f"week {TARGET_WEEK} spread_line coverage: {wk1_games['spread_line'].notna().mean():.0%}")
print(f"week {TARGET_WEEK} total_line coverage:  {wk1_games['total_line'].notna().mean():.0%}")
print(f"week {TARGET_WEEK} temp coverage (expect low -- weather isn't known months out): {wk1_games['temp'].notna().mean():.0%}")

sleeper_players = get_sleeper_players()
crosswalk = get_id_crosswalk()
league = get_sleeper_league(LEAGUE_ID, refresh=True)
print(f"\nleague {LEAGUE_ID}: season={league.get('season')} status={league.get('status')}")

weekly_features: 45,693 rows, seasons 2018-2025



2026 schedule: 272 rows, 16 week-1 games
week 1 spread_line coverage: 100%
week 1 total_line coverage:  100%
week 1 temp coverage (expect low -- weather isn't known months out): 0%



league 1389706592789733376: season=2026 status=pre_draft


## Data-availability check

This is a real, currently-live league — flagging plainly rather than assuming
anything about draft state or line availability.

In [4]:
if league.get("status") != "in_season" and league.get("status") != "post_season":
    print(f"NOTE: league status is '{league.get('status')}'.")
    if league.get("status") == "pre_draft":
        print("The draft has not happened yet -- real rosters are empty. \"Rostered\" "
              "scope will correctly come back empty; export will fall back to top-N "
              "free agents by projection only, which is the honest reflection of "
              "today's real state, not a bug. Re-running this notebook after the "
              "draft will pick up real rosters automatically, with no code change.")

assert wk1_games["spread_line"].notna().mean() > 0.9, (
    "Week 1 Vegas lines are mostly missing -- too early to build honest context "
    "features for this target week. Stop rather than export nulls dressed up as real context."
)

NOTE: league status is 'pre_draft'.
The draft has not happened yet -- real rosters are empty. "Rostered" scope will correctly come back empty; export will fall back to top-N free agents by projection only, which is the honest reflection of today's real state, not a bug. Re-running this notebook after the draft will pick up real rosters automatically, with no code change.


## Candidate pool

Built starting from this project's OWN historical data (gsis_id space), not
from Sleeper's player list — every QB/RB/WR/TE with at least one real row in
`weekly_features.parquet`, i.e. every player this model could possibly say
something honest about. Crosswalks to Sleeper for each candidate's current
team, reporting the match rate against the full position-eligible
population (not a pre-filtered one) — this is the real "crosswalk from
gsis_id at export time" check, reported here rather than assumed. Players
who have never played an NFL game (true rookies) have no historical row to
build features from and are honestly out of scope — not defaulted to a
league-average guess.

In [5]:
candidates, candidate_report = get_export_candidates(weekly_features, sleeper_players, crosswalk)
print("candidate report:", candidate_report)
print()
print(candidates["position"].value_counts())

candidate report: {'n_position_eligible': 1483, 'n_crosswalk_matched': 1468, 'n_with_current_team': 632, 'crosswalk_match_rate': 0.9898853674983142}

position
WR    249
RB    153
TE    140
QB     90
Name: count, dtype: int64


## Build target-week features (point-in-time safe)

Appends one stub row per candidate for the target week, then re-runs
`add_context_features`/`add_rolling_features` over the combined frame — see
`src/export.py`'s module docstring for why this needs no new feature logic.

In [6]:
combined_features = build_target_week_features(
    weekly_features, candidates, schedule_2026, TARGET_SEASON, TARGET_WEEK
)
stub_rows = combined_features[(combined_features["season"] == TARGET_SEASON) & (combined_features["week"] == TARGET_WEEK)]
print(f"combined frame: {len(combined_features):,} rows ({len(stub_rows)} target-week stub rows)")

# Sanity check on the mechanism itself: in-season rolling should be null
# (nothing played yet this season), prev_season_* should be populated for
# anyone who played in {XFP_SEASON}.
print(f"\nstub rows with null target_share_ewm3 (expected: all): "
      f"{stub_rows['target_share_ewm3'].isna().mean():.0%}")
print(f"stub rows with a real prev_season_target_share: "
      f"{stub_rows['prev_season_target_share'].notna().mean():.0%}")
print(f"stub rows games_played == 0 (expected: all): {(stub_rows['games_played'] == 0).mean():.0%}")

combined frame: 46,325 rows (632 target-week stub rows)

stub rows with null target_share_ewm3 (expected: all): 100%
stub rows with a real prev_season_target_share: 85%
stub rows games_played == 0 (expected: all): 100%


## Train final models and predict Week 1

One regression model (for `point`) and one q10/q90 CQR-calibrated quantile
pair (for `floor`/`ceiling`) per position, trained on all available history —
no fold is held out, since there's no ground truth yet for a week that
hasn't been played. No hyperparameter tuning, matching every other model in
this pipeline.

In [7]:
predictions = predict_target_week(combined_features, TARGET_SEASON, TARGET_WEEK)
print(f"predictions: {len(predictions)} rows")
print(predictions.groupby("position")[["point", "floor", "ceiling"]].describe().round(2))

assert (predictions["floor"] <= predictions["point"]).all()
assert (predictions["point"] <= predictions["ceiling"]).all()
print("\nfloor <= point <= ceiling holds for every prediction row.")

predictions: 632 rows
          point                                ... ceiling                                   
          count  mean   std   min   25%   50%  ...     std    min    25%    50%    75%    max
position                                       ...                                           
QB         90.0  9.27  5.06  2.40  5.87  6.69  ...    3.81  14.49  19.19  22.00  24.48  32.13
RB        153.0  5.34  3.55  1.98  3.13  4.05  ...    4.42   6.02   9.36  11.63  13.67  25.92
TE        140.0  2.94  1.71  1.39  1.62  2.41  ...    2.24   5.05   5.70   7.63   8.53  16.37
WR        249.0  4.69  2.68  2.04  2.73  3.88  ...    3.91   7.38   8.87  11.91  12.80  26.68

[4 rows x 24 columns]

floor <= point <= ceiling holds for every prediction row.


## Usage snapshot and season xFP summary

In [ ]:
usage = build_usage_snapshot(combined_features, TARGET_SEASON, TARGET_WEEK)
trend = build_trend_snapshot(combined_features, TARGET_SEASON, TARGET_WEEK)
xfp_summary = build_xfp_summary(weekly_features, XFP_SEASON)
# Per-week xfp for TARGET_SEASON's played weeks so far (Phase 8 round 3) --
# always TARGET_SEASON, unlike xfp_summary's XFP_SEASON fallback above,
# since the Weekly Production chart only ever plots TARGET_SEASON's own
# bars. Honestly empty at week 1 of a new season (nothing played yet).
weekly_xfp = build_weekly_xfp(weekly_features, TARGET_SEASON)
print(f"usage rows: {len(usage)}   trend rows: {len(trend)}   xfp_summary rows: {len(xfp_summary)} (season {XFP_SEASON})   "
      f"weekly_xfp rows: {len(weekly_xfp)} (season {TARGET_SEASON})")


Null-rate check for the trend block, reported plainly rather than assumed. Week 1 of a new season is expected to come back entirely null — every in-season `_ewm3` input is null by construction until the player has a real game this season (the same reason `target_share_ewm3` is already null at week 1 in the `usage` block above) — so this is a sanity check that the numbers match that expectation, not a surprise if they do.

In [9]:
from src.usage import TREND_SOURCE_FEATURES

trend_cols = (
    [f"trend_{f}_current" for f in TREND_SOURCE_FEATURES]
    + [f"{f}_trend_signal" for f in TREND_SOURCE_FEATURES]
    + [f"{f}_trend_direction" for f in TREND_SOURCE_FEATURES]
)
print(trend[trend_cols].isna().mean().round(3))


trend_target_share_current              1.0
trend_carry_share_current               1.0
trend_offense_pct_current               1.0
trend_rz_opportunity_share_current      1.0
target_share_trend_signal               1.0
carry_share_trend_signal                1.0
offense_pct_trend_signal                1.0
rz_opportunity_share_trend_signal       1.0
target_share_trend_direction            1.0
carry_share_trend_direction             1.0
offense_pct_trend_direction             1.0
rz_opportunity_share_trend_direction    1.0
dtype: float64


## Scope: real 2026 rosters, union top ~300 free agents by projection

Rostered players are never displaced by the top-N cutoff. Free agents are
ranked by this model's own `point` projection.

In [10]:
rosters_raw = get_sleeper_rosters(LEAGUE_ID, refresh=True)
rostered_sleeper_ids = {pid for r in rosters_raw for pid in (r.get("players") or [])}
print(f"real rostered sleeper_ids in league {LEAGUE_ID}: {len(rostered_sleeper_ids)}")

cw_lookup = crosswalk.dropna(subset=["sleeper_id", "gsis_id"]).drop_duplicates(subset=["sleeper_id"])
sleeper_to_gsis = dict(zip(cw_lookup["sleeper_id"], cw_lookup["gsis_id"]))
rostered_gsis_ids = {sleeper_to_gsis[sid] for sid in rostered_sleeper_ids if sid in sleeper_to_gsis}
print(f"rostered sleeper_ids with a known gsis_id: {len(rostered_gsis_ids)}")

scoped_predictions, scope_report = get_export_scope(rostered_gsis_ids, predictions, top_n=TOP_N_FREE_AGENTS)
print(scope_report)

real rostered sleeper_ids in league 1389706592789733376: 0
rostered sleeper_ids with a known gsis_id: 0
{'n_rostered': 0, 'n_top_n': 300, 'n_total': 300}


## Radar percentiles (Phase 4)

Six axes per position, each an already-computed Family 1-4 `_s2d` (season-to-date)
column — see `src/export.py`'s `RADAR_METRICS` for the exact list and why `_s2d`
rather than `_ewm3`. Percentiled against this league's REAL startable pool
(`position_starter_counts()`, a direct Python port of `index.html`'s
`positionStarterCount()`) — top-N players per position by season-to-date total
`custom_points`, restricted to players who themselves clear the games-played
floor. A player short on games gets an honest `eligible: False` object instead
of a partial/misleading radar shape.

In [ ]:
radar = build_radar_snapshot(
    combined_features, TARGET_SEASON, TARGET_WEEK, league["roster_positions"], len(rosters_raw)
)
n_eligible = sum(1 for r in radar.values() if r["eligible"])
print(f"radar: {n_eligible}/{len(radar)} candidates eligible (>= games-played floor)")
print(f"expected at week 1 of a new season: 0 eligible (games_played == 0 for every stub row)")

## Field heatmap zones (Phase 5)

Derived directly from real play-by-play, not from `weekly_features` — real
targets (`src.usage.receiving_zone_plays`/`passing_zone_plays`) and real
carries (`rushing_zone_plays`), zoned by air-yards depth x field position
(receivers) or direction x field position (runners) or location x depth
(QBs), all using `src.usage`'s HEATMAP_* bins (shaped for a display grid,
not xFP's rate-estimation bins). Same games-played eligibility floor as
radar; a zone below `HEATMAP_SPARSE_THRESHOLD` real plays is flagged
`sparse` rather than dropped or merged — see `build_heatmap_snapshot`'s
docstring.

In [ ]:
# pbp for this season only -- get_pbp caches to data/raw/, and
# weekly_features.parquet was already built from the FULL multi-season pbp
# earlier (03_usage_features.ipynb), so this is a light single-season fetch,
# not a re-download of everything.
#
# nflreadpy.load_pbp() raises ValueError for a season that hasn't started
# yet (a client-side range check, not a network 404) -- the real case for
# TARGET_SEASON=2026 right now. build_heatmap_snapshot treats a completely
# empty pbp frame as "nothing to zone yet," which is correct here: 5+ games
# played (the eligibility floor) is impossible without pbp for those games
# existing, so every candidate is ineligible either way in this scenario.
try:
    pbp_current = get_pbp([TARGET_SEASON])
except ValueError as e:
    if "must be between" not in str(e):
        raise
    print(f"season {TARGET_SEASON} has no published play-by-play yet -- heatmap zones will be empty.")
    pbp_current = pd.DataFrame()
print(f"pbp_current: {len(pbp_current):,} rows for season {TARGET_SEASON}")

heatmap = build_heatmap_snapshot(combined_features, pbp_current, TARGET_SEASON, TARGET_WEEK)
n_eligible = sum(1 for h in heatmap.values() if h["eligible"])
print(f"heatmap: {n_eligible}/{len(heatmap)} candidates eligible (>= games-played floor)")
print(f"expected at week 1 of a new season: 0 eligible (games_played == 0 for every stub row)")

## Assemble and validate the JSON payload

Crosswalks gsis_id → sleeper_id at this final step (everything upstream is
in gsis_id space) and reports the match rate plainly, rather than assuming
it.

In [ ]:
payload, crosswalk_report = assemble_player_advanced_stats(
    scoped_predictions, usage, trend, xfp_summary, weekly_xfp, radar, heatmap, crosswalk,
    TARGET_SEASON, TARGET_WEEK, XFP_SEASON, SEASONS_TRAINED, MODEL_VERSION,
)
print("crosswalk match rate:", crosswalk_report)

validation_report = validate_export(payload, crosswalk)
print("validation:", validation_report)

## Write output and report size

In [12]:
out_path = DATA_OUTPUT / "player_advanced_stats.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    json.dump(payload, f)

size_bytes = out_path.stat().st_size
print(f"wrote {out_path}")
print(f"size: {size_bytes:,} bytes ({size_bytes / 1_000_000:.3f} MB)")
print(f"under 2 MB target: {size_bytes < 2_000_000}")

# Round-trip parse check
with open(out_path) as f:
    reparsed = json.load(f)
assert reparsed == payload
print(f"\nJSON round-trips cleanly. {len(payload['players'])} players in the export.")

wrote C:\Users\rohbh\Claude Projects\fanteasy-notebook\data\output\player_advanced_stats.json
size: 219,330 bytes (0.219 MB)
under 2 MB target: True

JSON round-trips cleanly. 300 players in the export.


## Notes for next time

- `trend` (Phase 3'), `radar` (Phase 4, percentile profile), and `heatmap`
  (Phase 5, field zones) are all now real sibling keys alongside
  `projection`/`usage`/`xfp` — every phase after Phase 3' added its own new
  sibling key with no restructuring of the ones before it.
- `meta.performance`/`meta.caveats` are hardcoded constants in `src/export.py`
  (`PERFORMANCE_BY_POSITION`, `CAVEATS`), reused from the already-published
  Phase 6 findings — re-derive them there (not here) if the model is ever
  retrained.
- K/DST are out of scope for this export, same as the projection model
  itself — the dashboard keeps showing Sleeper's own K/DST numbers directly.
- `radar` and `heatmap` both come back `eligible: False` for literally
  everyone at week 1 of a new season (`games_played == 0` for every stub
  row, same reason `usage`/`trend` are null then) — this is expected, not a
  bug, and both will populate naturally from week 6 onward once real
  players clear `MIN_GAMES_FOR_TREND`, same pattern as `trend`'s own week-1
  note above.